In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
data = pd.read_csv('data/Africa towers.csv')

In [4]:
data.head()

,Unnamed: 0,radio,MCC,MNC,TAC,CID,unit,LON,LAT,RANGE,SAM,changeable,created,updated,averageSignal,Country,Network,Continent
0,12847759,GSM,602,3,21333,25372,0,31.056510,29.998215,23897,10,1,1459715136,1465348064,0,Egypt,Etisalat,Africa
1,12847760,GSM,602,3,21362,23224,0,31.373520,29.839554,1000,4,1,1459813831,1474212997,0,Egypt,Etisalat,Africa
2,12847761,GSM,602,3,22533,5031,0,31.160660,29.998856,1000,1,1,1459695955,1459695955,0,Egypt,Etisalat,Africa
3,12847762,GSM,602,3,22202,40686,0,31.501236,30.592117,1000,1,1,1459681907,1459681907,0,Egypt,Etisalat,Africa
4,12847763,GSM,602,3,21333,25376,0,31.277390,30.095673,1000,2,1,1459715136,1460478702,0,Egypt,Etisalat,Africa


In [10]:
drop_columns=['unit','changeable','averageSignal','Continent']
data=data.drop(drop_columns,axis=1)

In [11]:
data.head()

,Unnamed: 0,radio,MCC,MNC,TAC,CID,LON,LAT,RANGE,SAM,created,updated,Country,Network
0,12847759,GSM,602,3,21333,25372,31.056510,29.998215,23897,10,1459715136,1465348064,Egypt,Etisalat
1,12847760,GSM,602,3,21362,23224,31.373520,29.839554,1000,4,1459813831,1474212997,Egypt,Etisalat
2,12847761,GSM,602,3,22533,5031,31.160660,29.998856,1000,1,1459695955,1459695955,Egypt,Etisalat
3,12847762,GSM,602,3,22202,40686,31.501236,30.592117,1000,1,1459681907,1459681907,Egypt,Etisalat
4,12847763,GSM,602,3,21333,25376,31.277390,30.095673,1000,2,1459715136,1460478702,Egypt,Etisalat


In [12]:
data['Country'].unique()

<StringArray>
[                       'Egypt',                     'Ethiopia',
                      'Nigeria',                      'Morocco',
                      'Senegal',                      'Algeria',
                       'Malawi',                        'Ghana',
                      'Tunisia',                  'Ivory Coast',
                 'South Africa',                        'Sudan',
                     'Zimbabwe',                       'Zambia',
                        'Kenya',                         'Mali',
                     'Tanzania', 'Democratic Republic of Congo',
                   'Mozambique',                     'Cameroon',
                    'Mauritius',                       'Angola',
                      'Namibia',                      'Mayotte',
                        'Benin',                         'Chad',
                 'Burkina Faso',                   'Cape Verde',
                      'Reunion',                        'Gabon',
           

In [13]:
data = data[data['Country'] == 'Egypt']


In [14]:
# rename column Unnamed: 0
data = data.rename(columns={'Unnamed: 0': ' ID'})

In [15]:
data.head()

,ID,radio,MCC,MNC,TAC,CID,LON,LAT,RANGE,SAM,created,updated,Country,Network
0,12847759,GSM,602,3,21333,25372,31.056510,29.998215,23897,10,1459715136,1465348064,Egypt,Etisalat
1,12847760,GSM,602,3,21362,23224,31.373520,29.839554,1000,4,1459813831,1474212997,Egypt,Etisalat
2,12847761,GSM,602,3,22533,5031,31.160660,29.998856,1000,1,1459695955,1459695955,Egypt,Etisalat
3,12847762,GSM,602,3,22202,40686,31.501236,30.592117,1000,1,1459681907,1459681907,Egypt,Etisalat
4,12847763,GSM,602,3,21333,25376,31.277390,30.095673,1000,2,1459715136,1460478702,Egypt,Etisalat


In [16]:
data.shape

(236268, 14)

In [17]:
data.info()

<class 'pandas.DataFrame'>
Index: 236268 entries, 0 to 2340146
Data columns (total 14 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0    ID      236268 non-null  int64  
 1   radio    236268 non-null  str    
 2   MCC      236268 non-null  int64  
 3   MNC      236268 non-null  int64  
 4   TAC      236268 non-null  int64  
 5   CID      236268 non-null  int64  
 6   LON      236268 non-null  float64
 7   LAT      236268 non-null  float64
 8   RANGE    236268 non-null  int64  
 9   SAM      236268 non-null  int64  
 10  created  236268 non-null  int64  
 11  updated  236268 non-null  int64  
 12  Country  236268 non-null  str    
 13  Network  236268 non-null  str    
dtypes: float64(2), int64(9), str(3)
memory usage: 27.0 MB


In [18]:
data.columns = data.columns.str.strip()


In [35]:

data = data[data["Country"].astype(str).str.strip() == "Egypt"].reset_index(drop=True)

# ------------------------------------------------------
# 1. DERIVED KPIs
# ------------------------------------------------------

# Tower Density = number of towers / area
# --> You need area, but we simulate by grouping per city/region later

# Drop Rate (simulate: RANGE column can represent traffic or attempts)
np.random.seed(42)
data["drop_calls"] = np.random.randint(0, 50, len(data))
data["total_calls"] = data["RANGE"] + data["drop_calls"]
data["drop_rate"] = (data["drop_calls"] / data["total_calls"]) * 100

# Average Load per Tower = total_calls / tower count (per MCC/MNC area)
tower_counts = data.groupby(["MCC", "MNC"])["ID"].transform("count")
data["avg_load"] = data["total_calls"] / tower_counts

# QoE Score (0–5 scale) → composite measure
data["signal_strength"] = np.random.randint(-105, -60, len(data))
data["speed"] = np.random.uniform(1, 80, len(data))          # Mbps
data["latency"] = np.random.uniform(10, 200, len(data))      # ms

# Simple QoE formula
data["QoE"] = (
    ((data["signal_strength"] + 110) / 50) * 0.4 +  # normalize 0-1
    (data["speed"] / 80) * 0.4 +
    (1 - (data["latency"] / 200)) * 0.2
) * 5

# Coverage Gap: distance > 2 km
# Simple approximation using RANGE
data["coverage_gap"] = data["RANGE"] > 2000

# ------------------------------------------------------
# 2. CLASSIFICATIONS
# ------------------------------------------------------

# Signal Quality
def classify_signal(dBm):
    if -70 <= dBm <= -60:
        return "Excellent"
    elif -80 <= dBm < -70:
        return "Good"
    elif -90 <= dBm < -80:
        return "Fair"
    elif -100 <= dBm < -90:
        return "Poor"
    else:
        return "Very Poor"

data["signal_quality"] = data["signal_strength"].apply(classify_signal)


# Tower Status
def tower_status(dr):
    if dr < 20: return "Healthy"
    elif dr < 40: return "Warning"
    elif dr < 60: return "Critical"
    else: return "Failed"

data["tower_status"] = data["drop_rate"].apply(tower_status)


# Priority Level
def assign_priority(status):
    if "Failed" in status or "Critical" in status:
        return "P1"
    elif "Warning" in status:
        return "P2"
    else:
        return "P3"

data["priority"] = data["tower_status"].apply(assign_priority)

data.head()

,ID,radio,MCC,MNC,TAC,CID,LON,LAT,RANGE,SAM,...,priority,maintenance_type,created_dt,updated_dt,maintenance_date,labor_cost_egp,parts_cost_egp,downtime_hours,vendor,notes
0,12847759,GSM,602,3,21333,25372,31.056510,29.998215,23897,10,...,P3,preventive,2016-04-03 20:25:36,2016-06-08 01:07:44,2016-05-16 18:43:33.432938020,3527,582,2.2,FiberMisr,
1,12847760,GSM,602,3,21362,23224,31.373520,29.839554,1000,4,...,P3,predictive,2016-04-04 23:50:31,2016-09-18 15:36:37,2016-07-29 07:03:01.916462872,5440,1423,5.9,Benya,
2,12847761,GSM,602,3,22533,5031,31.160660,29.998856,1000,1,...,P3,predictive,2016-04-03 15:05:55,2016-04-03 15:05:55,2016-04-03 15:05:55.000000000,6024,2104,1.7,Nokia,Replaced sector antenna
3,12847762,GSM,602,3,22202,40686,31.501236,30.592117,1000,1,...,P3,preventive,2016-04-03 11:11:47,2016-04-03 11:11:47,2016-04-03 11:11:47.000000000,2383,1328,0.7,Ericsson,Replaced sector antenna
4,12847763,GSM,602,3,21333,25376,31.277390,30.095673,1000,2,...,P3,preventive,2016-04-03 20:25:36,2016-04-12 16:31:42,2016-04-10 10:19:37.189017394,2704,2395,0.7,FiberMisr,


In [20]:
data.head()

,ID,radio,MCC,MNC,TAC,CID,LON,LAT,RANGE,SAM,...,drop_rate,avg_load,signal_strength,speed,latency,QoE,coverage_gap,signal_quality,tower_status,priority
0,12847759,GSM,602,3,21333,25372,31.056510,29.998215,23897,10,...,0.158763,0.361813,-73,16.383672,42.638550,2.676399,True,Good,Healthy,P3
1,12847760,GSM,602,3,21362,23224,31.373520,29.839554,1000,4,...,2.723735,0.015540,-102,71.456456,23.668258,2.988070,False,Very Poor,Healthy,P3
2,12847761,GSM,602,3,22533,5031,31.160660,29.998856,1000,1,...,1.380671,0.015328,-67,13.757702,43.686967,2.845508,False,Excellent,Healthy,P3
3,12847762,GSM,602,3,22202,40686,31.501236,30.592117,1000,1,...,4.030710,0.015751,-96,17.359251,129.737548,1.345294,False,Poor,Healthy,P3
4,12847763,GSM,602,3,21333,25376,31.277390,30.095673,1000,2,...,0.695134,0.015222,-105,43.515775,164.099024,1.467399,False,Very Poor,Healthy,P3


In [21]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 236268 entries, 0 to 236267
Data columns (total 26 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   ID               236268 non-null  int64  
 1   radio            236268 non-null  str    
 2   MCC              236268 non-null  int64  
 3   MNC              236268 non-null  int64  
 4   TAC              236268 non-null  int64  
 5   CID              236268 non-null  int64  
 6   LON              236268 non-null  float64
 7   LAT              236268 non-null  float64
 8   RANGE            236268 non-null  int64  
 9   SAM              236268 non-null  int64  
 10  created          236268 non-null  int64  
 11  updated          236268 non-null  int64  
 12  Country          236268 non-null  str    
 13  Network          236268 non-null  str    
 14  drop_calls       236268 non-null  int32  
 15  total_calls      236268 non-null  int64  
 16  drop_rate        236268 non-null  float64
 17  av

In [34]:
data['vendor'].value_counts()

vendor
Nokia               33930
Local Contractor    33828
Ericsson            33792
Huawei              33763
FiberMisr           33739
Benya               33711
ZTE                 33505
Name: count, dtype: int64

In [23]:
np.random.seed(42)

# 1. Maintenance type
maintenance_types = ["preventive", "predictive", "emergency"]
maintenance_prob = [0.60, 0.25, 0.15]
data["maintenance_type"] = np.random.choice(
    maintenance_types, size=len(data), p=maintenance_prob
)

#2 Convert created & updated to datetime
data["created_dt"] = pd.to_datetime(data["created"], unit="s")
data["updated_dt"] = pd.to_datetime(data["updated"], unit="s")

# Ensure updated >= created (just in case)
mask = data["updated_dt"] < data["created_dt"]
data.loc[mask, "updated_dt"] = data.loc[mask, "created_dt"] + pd.Timedelta(days=30)

# Maintenance type
maintenance_types = ["preventive", "predictive", "emergency"]
maintenance_prob = [0.60, 0.25, 0.15]
data["maintenance_type"] = np.random.choice(
    maintenance_types, size=len(data), p=maintenance_prob
)

def generate_maintenance_date(row):
    start = row["created_dt"]
    end = row["updated_dt"]

    # Total seconds between created and updated
    total_seconds = int((end - start).total_seconds())

    if total_seconds <= 0:
        return start  # fallback

    if row["maintenance_type"] == "preventive":
        # Preventive: close to updated (last 3–6 months)
        offset_seconds = np.random.uniform(total_seconds * 0.6, total_seconds * 1.0)

    elif row["maintenance_type"] == "predictive":
        # Predictive: middle of the lifecycle
        offset_seconds = np.random.uniform(total_seconds * 0.3, total_seconds * 0.7)

    else:  # emergency
        # Emergency: anywhere randomly
        offset_seconds = np.random.uniform(0, total_seconds)

    return start + pd.Timedelta(seconds=offset_seconds)

# Generate the maintenance date
data["maintenance_date"] = data.apply(generate_maintenance_date, axis=1)

# 3. Costs based on maintenance type
def gen_labor_cost(mt):
    if mt == "preventive": return np.random.randint(1500, 4000)
    if mt == "predictive": return np.random.randint(3000, 7000)
    return np.random.randint(5000, 15000)

def gen_parts_cost(mt):
    if mt == "preventive": return np.random.randint(500, 2500)
    if mt == "predictive": return np.random.randint(1000, 5000)
    return np.random.randint(3000, 20000)

data["labor_cost_egp"]  = data["maintenance_type"].apply(gen_labor_cost)
data["parts_cost_egp"] = data["maintenance_type"].apply(gen_parts_cost)

# 4. Downtime hours
def gen_downtime(mt):
    if mt == "preventive": return round(np.random.uniform(0.5, 3), 1)
    if mt == "predictive": return round(np.random.uniform(1, 6), 1)
    return round(np.random.uniform(4, 24), 1)

data["downtime_hours"] = data["maintenance_type"].apply(gen_downtime)

# 5. Vendors
vendors = ["Huawei", "Ericsson", "Nokia", "ZTE", "FiberMisr", "Benya", "Local Contractor"]
data["vendor"] = np.random.choice(vendors, size=len(data))

# 6. Notes
notes_list = [
    "Replaced sector antenna",
    "Battery replaced",
    "Fiber cut repaired",
    "Power system failure",
    "Software upgrade",
    "Transmission issue",
    "Cooling system maintenance",
    ""
]
data["notes"] = np.random.choice(notes_list, size=len(data), p=[0.08,0.08,0.08,0.08,0.08,0.08,0.02,0.5])

In [24]:
data.head()

,ID,radio,MCC,MNC,TAC,CID,LON,LAT,RANGE,SAM,...,priority,maintenance_type,created_dt,updated_dt,maintenance_date,labor_cost_egp,parts_cost_egp,downtime_hours,vendor,notes
0,12847759,GSM,602,3,21333,25372,31.056510,29.998215,23897,10,...,P3,preventive,2016-04-03 20:25:36,2016-06-08 01:07:44,2016-05-16 18:43:33.432938020,3527,582,2.2,FiberMisr,
1,12847760,GSM,602,3,21362,23224,31.373520,29.839554,1000,4,...,P3,predictive,2016-04-04 23:50:31,2016-09-18 15:36:37,2016-07-29 07:03:01.916462872,5440,1423,5.9,Benya,
2,12847761,GSM,602,3,22533,5031,31.160660,29.998856,1000,1,...,P3,predictive,2016-04-03 15:05:55,2016-04-03 15:05:55,2016-04-03 15:05:55.000000000,6024,2104,1.7,Nokia,Replaced sector antenna
3,12847762,GSM,602,3,22202,40686,31.501236,30.592117,1000,1,...,P3,preventive,2016-04-03 11:11:47,2016-04-03 11:11:47,2016-04-03 11:11:47.000000000,2383,1328,0.7,Ericsson,Replaced sector antenna
4,12847763,GSM,602,3,21333,25376,31.277390,30.095673,1000,2,...,P3,preventive,2016-04-03 20:25:36,2016-04-12 16:31:42,2016-04-10 10:19:37.189017394,2704,2395,0.7,FiberMisr,


In [25]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 236268 entries, 0 to 236267
Data columns (total 35 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   ID                236268 non-null  int64         
 1   radio             236268 non-null  str           
 2   MCC               236268 non-null  int64         
 3   MNC               236268 non-null  int64         
 4   TAC               236268 non-null  int64         
 5   CID               236268 non-null  int64         
 6   LON               236268 non-null  float64       
 7   LAT               236268 non-null  float64       
 8   RANGE             236268 non-null  int64         
 9   SAM               236268 non-null  int64         
 10  created           236268 non-null  int64         
 11  updated           236268 non-null  int64         
 12  Country           236268 non-null  str           
 13  Network           236268 non-null  str           
 14  drop_calls     

In [33]:
print(data['created'])
print(data['created_dt'])



0         1459715136
1         1459813831
2         1459695955
3         1459681907
4         1459715136
             ...    
236263    1691181486
236264    1691182077
236265    1691298625
236266    1691308131
236267    1691328636
Name: created, Length: 236268, dtype: int64
0        2016-04-03 20:25:36
1        2016-04-04 23:50:31
2        2016-04-03 15:05:55
3        2016-04-03 11:11:47
4        2016-04-03 20:25:36
                 ...        
236263   2023-08-04 20:38:06
236264   2023-08-04 20:47:57
236265   2023-08-06 05:10:25
236266   2023-08-06 07:48:51
236267   2023-08-06 13:30:36
Name: created_dt, Length: 236268, dtype: datetime64[s]


In [27]:
print(data['updated'])
print(data['updated_dt'])

0         1465348064
1         1474212997
2         1459695955
3         1459681907
4         1460478702
             ...    
236263    1691181951
236264    1691182077
236265    1691298625
236266    1691308131
236267    1691328636
Name: updated, Length: 236268, dtype: int64
0        2016-06-08 01:07:44
1        2016-09-18 15:36:37
2        2016-04-03 15:05:55
3        2016-04-03 11:11:47
4        2016-04-12 16:31:42
                 ...        
236263   2023-08-04 20:45:51
236264   2023-08-04 20:47:57
236265   2023-08-06 05:10:25
236266   2023-08-06 07:48:51
236267   2023-08-06 13:30:36
Name: updated_dt, Length: 236268, dtype: datetime64[s]


In [28]:
data['maintenance_date']

0        2016-05-16 18:43:33.432938020
1        2016-07-29 07:03:01.916462872
2        2016-04-03 15:05:55.000000000
3        2016-04-03 11:11:47.000000000
4        2016-04-10 10:19:37.189017394
                      ...             
236263   2023-08-04 20:45:32.688455766
236264   2023-08-04 20:47:57.000000000
236265   2023-08-06 05:10:25.000000000
236266   2023-08-06 07:48:51.000000000
236267   2023-08-06 13:30:36.000000000
Name: maintenance_date, Length: 236268, dtype: datetime64[ns]

In [29]:
data.to_csv("FULL_telecom_dataset.csv", index=False)


In [30]:
data.shape

(236268, 35)